## **Fase 1 - Pronóstico Bivariado: Demanda No Regulada y Precio de Bolsa**

El objetivo de esta fase es predecir simultáneamente la Demanda de energía eléctrica No Regulada y el Precio del Mercado de energía a nivel nacional, hora a hora. Contar con un pronóstico preciso de ambas variables es clave para la siguiente fase, ya que es el insumo principal para alimentar la posterior función de optimización.

### **1. Estructura del Conjunto de Datos**
La matriz de información ha sido consolidada mediante un proceso de extracción y agregación espacial desde el mercado mayorista (XM).

* **Temporalidad:** 1 de enero de 2022 al 31 de agosto de 2026 (1.704 días).
* **Granularidad:** Horaria.
* **Dimensión del tensor:** $40.896 \times 5$ observaciones.
* **Variable Objetivo ($Y_t$):** Vector bidimensional continuo que contiene Demanda_No_Regulada_kWh y Precio_Bolsa_COP.
* **Covariables Exógenas ($X_t$):** Precio Promedio Contratos, Aportes Hídricos y Volumen Útil.

La inclusión de estas variables exógenas se fundamenta en que el consumo energético industrial no obedece exclusivamente a su propia inercia histórica, sino que responde de forma dinámica a estímulos del mercado y del clima.

* **Bolsa y Contratos:** Capturan la elasticidad de los grandes consumidores. Ante picos tarifarios, la industria suele reaccionar desplazando turnos de producción hacia horas más baratas o activando plantas de autogeneración, lo que altera la curva de demanda típica.
* **Volumen Útil y Aportes:** Dado que la matriz eléctrica colombiana es furtemente hidráulica, el estado de los embalses opera como un indicador adelantado de escasez (ej. Fenómeno de El Niño), condicionando estructuralmente la oferta y el costo de la energía.

Al integrar estas covariables en el tensor de entrenamiento, se dota a la red neuronal de dimensiones adicionales para que aprenda las relaciones no lineales entre el clima, la economía y la industria, permitiéndole anticipar variaciones en el consumo que serían imposibles de predecir usando únicamente enfoques autorregresivos tradicionales.

### **2. Formulación Matemática**
La arquitectura de Deep Learning se define como una función no lineal $f$ parametrizada por los pesos $\Theta$. El modelo mapea una ventana histórica de observaciones conjuntas con longitud $k$ y emite un vector bidimensional de predicción para el horizonte futuro $h$:

$$\widehat{Y}_{t+h} = \begin{bmatrix} \widehat{D}_{t+h} \\ \widehat{P}^{Bolsa}_{t+h} \end{bmatrix} = f(Y_{t-k:t}, X_{t-k:t} ; \Theta)$$

Donde $\widehat{D}_{t+h}$ es la demanda industrial proyectada y $\widehat{P}^{Bolsa}_{t+h}$ es el precio del mercado spot proyectado.

### **3. Arquitectura General y Función de Pérdida**
El modelo minimizará el Error Cuadrático Medio (MSE) sobre el vector de predicción bidimensional. 

Esta función de pérdida se selecciona porque penaliza cuadráticamente las desviaciones grandes, una característica crítica en la planeación energética industrial para evitar subestimar picos severos de consumo o explosiones abruptas en el precio tarifario:

$$MSE = \frac{1}{N} \sum_{i=1}^{N} \left\Vert Y_i - \widehat{Y}_i \right\Vert_2^2$$

Donde:
* $N$: Número total de muestras en el lote.
* $Y_i$: Vector bidimensional con los valores reales observados en el instante de predicción.
* $\widehat{Y}_i$: Vector bidimensional predicho por la red neuronal.
* $\Vert \cdot \Vert_2^2$: Norma L2 al cuadrado (distancia euclidiana al cuadrado entre los vectores real y predicho).

## **Fase 2 - Precios y Optimización de Compras de Energía**

Una vez obtenido el vector de demanda proyectada de la Fase 1, el objetivo de la Fase 2 es estructurar la estrategia óptima de abastecimiento. El problema consiste en determinar la asignación ideal del portafolio, qué cantidad de la energía requerida debe adquirirse en el Mercado de Corto Plazo (Bolsa) y qué cantidad debe cubrirse mediante Contratos Bilaterales, minimizando el costo total de adquisición frente a la volatilidad tarifaria.

Debido a la dinámica real del mercado eléctrico los contratos bilaterales se negocian con meses o años de anticipación. Por tanto, el algoritmo no tomaría decisiones de compra de contratos nuevos en tiempo real, sino que gestiona un contrato preexistente, decidiendo hora a hora cuánta energía usar de dicho contrato y cuánta dejar expuesta a la bolsa.

### **1. Variables y Parámetros del Modelo**
Para cada periodo $t$ dentro del horizonte temporal $T$, las predicciones generadas en la Fase 1 ($\widehat{D}_t$ y $\widehat{P}^{Bolsa}_t$) actúan como los parámetros de entrada para el algoritmo de decisión.

Se define el vector de precios ($P_t$) y el vector de variables de decisión ($x_t$):

$$P_t = \begin{bmatrix} \widehat{P}^{Bolsa}_t \\ P^{Contrato}_t \end{bmatrix}, \quad x_t = \begin{bmatrix} x_{1,t} \\ x_{2,t} \end{bmatrix}$$

*   $x_{1,t}$: Cantidad de energía a adquirir en el mercado spot (bolsa).
*   $x_{2,t}$: Cantidad de energía a despachar del contrato bilateral preexistente.
*   $P^{Contrato}_t$: Es un parámetro fijo y conocido dictado por el contrato vigente

### **2. Formulación Matemática de la Función Objetivo**
Definimos el vector global de precios $P \in \mathbb{R}^{2T}$ y el vector de decisión global $X \in \mathbb{R}^{2T}$.

La función objetivo busca minimizar el gasto económico total calculando el producto punto entre el vector de precios y las decisiones de abastecimiento:

$$\min_{X} P^T X$$

### **3. Restricciones de Factibilidad**
El algoritmo de optimización debe respetar la física del negocio y las limitaciones comerciales preestablecidas.

*   **Restricción1 (Balance de Energía):** La suma de la energía nominada en contrato y la comprada en bolsa debe igualar exactamente la proyección de demanda de la red neuronal ($\widehat{D} \in \mathbb{R}^{T}$). Matricialmente se expresa como un sistema de ecuaciones lineales $AX = b$:

$$A X = \widehat{D}$$

Donde $A \in \mathbb{R}^{T \times 2T}$ es una matriz de coeficientes (basada en el vector transpuesto de unos) que suma los valores de bolsa y contrato correspondientes a cada periodo temporal.

*   **Restricción2 (Capacidad del Contrato):** Dado que los contratos bilaterales ya están firmados, tienen una capacidad máxima ($C_{max}$) establecida legalmente. La energía nominada del contrato no puede exceder este límite físico en ninguna hora:

$$x_{2,t} \le C_{max} \quad \forall t \in T$$

*   **Restricción3 (No Negatividad):** Para impedir algorítmicamente que el optimizador sugiera ventas anómalas de energía (valores negativos) buscando reducir costos, se acota el límite inferior:

$$X \ge 0$$